In [1]:
pip install opencv-python numpy matplotlib

   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.1 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.1 MB 2.6 MB/s eta 0:00:03
   ------- -------------------------------- 1.6/8.1 MB 2.9 MB/s eta 0:00:03
   ------------- -------------------------- 2.6/8.1 MB 3.5 MB/s eta 0:00:02
   ------------------- -------------------- 3.9/8.1 MB 4.1 MB/s eta 0:00:02
   ---------------------------- ----------- 5.8/8.1 MB 4.8 MB/s eta 0:00:01
   ---------------------------------------  7.9/8.1 MB 5.7 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 5.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ------------------- -------------------- 1.0/2.2 MB 4.6 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 6.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ------------------------------- -----

In [6]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 이미지 불러오기
image_path = 'KakaoTalk_light.jpg'
image = cv2.imread(image_path)
if image is None:
    raise ValueError("이미지를 불러올 수 없습니다. 경로를 확인하세요.")

image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# === 신호등 좌표 지정 ===
traffic_light = (100, 200, 40, 100)  # 실제 이미지 범위에 맞게 수정하세요
x, y, w, h = traffic_light

# === 이미지 크기 확인 및 ROI 유효성 검사 ===
img_h, img_w = image.shape[:2]
if x < 0 or y < 0 or x + w > img_w or y + h > img_h:
    raise ValueError(f"ROI 범위가 잘못되었습니다. 이미지 크기: ({img_w}, {img_h}), "
                     f"ROI: ({x}, {y}, {w}, {h})")

# === ROI 설정 ===
roi = image[y:y+h, x:x+w]
if roi.size == 0:
    raise ValueError("ROI가 비어 있습니다. 좌표를 다시 확인하세요.")

roi_hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

# === 1. 초록불 감지 ===
lower_green = np.array([40, 40, 40])
upper_green = np.array([90, 255, 255])
mask_green = cv2.inRange(roi_hsv, lower_green, upper_green)
green_ratio = np.sum(mask_green > 0) / (w * h)

if green_ratio > 0.03:
    cv2.rectangle(image_rgb, (x, y), (x+w, y+h), (0, 255, 0), 2)
    cv2.putText(image_rgb, 'Green Light', (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # === 2. 화살표 검출 (초록색 내부) ===
    green_part = cv2.bitwise_and(roi, roi, mask=mask_green)
    green_gray = cv2.cvtColor(green_part, cv2.COLOR_BGR2GRAY)
    _, arrow_thresh = cv2.threshold(green_gray, 50, 255, cv2.THRESH_BINARY)

    arrow_contours, _ = cv2.findContours(arrow_thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for acnt in arrow_contours:
        approx = cv2.approxPolyDP(acnt, 0.03 * cv2.arcLength(acnt, True), True)
        if len(approx) >= 5:
            cv2.putText(image_rgb, 'Green Arrow', (x, y + h + 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            break

# === 시각화 ===
plt.figure(figsize=(10, 5))
plt.title('Green Light and Arrow Detection')
plt.imshow(image_rgb)
plt.axis('off')
plt.show()

ValueError: ROI 범위가 잘못되었습니다. 이미지 크기: (972, 178), ROI: (100, 200, 40, 100)